[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/appars/codepilot-colab/blob/main/stage1_hello_groq/Stage1_Hello_Groq.ipynb)

> **Click the badge above to open this notebook in Google Colab.**
> Or go directly: https://colab.research.google.com/github/appars/codepilot-colab/blob/main/stage1_hello_groq/Stage1_Hello_Groq.ipynb

# 🤖 Stage 1 — Hello Groq
**CodePilot AI Studio | Module 4 | Agentic AI in Software Engineering**
**Instructor: Prof. Apparsamy Perumal**

---

## What You Will Learn
- What an **LLM** (Large Language Model) is
- What **Groq** is — a free, fast cloud AI service (responses in 1-3 seconds!)
- What **Llama 3** is — the open-source AI model we use (by Meta, free)
- What **LangChain** is — the Python framework that connects your code to the AI
- How to write a **prompt** and get a **response**

## The Big Picture
```
Your Python Code
      ↓
  LangChain          ← the connector (like a USB cable between Python and AI)
      ↓
  Groq API           ← the fast cloud engine (runs on special AI hardware)
      ↓
  Llama 3 (8B)       ← the AI brain (8 billion parameters, open source)
      ↓
  Response           ← back to your Python code in 1-3 seconds
```

> **Real-world analogy:** LangChain is the waiter. Groq is the fast kitchen. Llama 3 is the chef.

---
## How to Use This Notebook
1. Run cells **one by one** using the ▶ button next to each cell
2. Or press **Ctrl+F9** to run ALL cells at once
3. Read every comment inside the code — they explain WHY not just WHAT
4. Try the experiments in the last cells

⏱ **Expected time: 15 minutes**

## Step 1 — Add Your Groq API Key (One Time Setup)

### Option A — Using Colab Secrets (Recommended!)
Store your key ONCE in Colab Secrets and it works in ALL notebooks automatically:
1. Click the **🔑 key icon** in the left sidebar (or go to Tools → Secrets)
2. Click **'Add new secret'**
3. Name: `GROQ_API_KEY`  (must be exactly this name)
4. Value: paste your key (looks like `gsk_xxxx...`)
5. Toggle **'Notebook access'** to ON
6. Come back and run this cell

### Option B — Paste directly (quick one-time use)
If you do not want to use Secrets, just paste your key in the code cell below.

> Get your free key at **https://console.groq.com** → API Keys → Create API Key

In [ ]:
# ── GROQ API KEY SETUP ───────────────────────────────────────
# This cell tries Colab Secrets first (recommended).
# If not found, falls back to manual paste below.

import os

# ── METHOD 1: Colab Secrets (store once, works in all notebooks)
# If you added GROQ_API_KEY in the Secrets panel, this will find it.
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    if GROQ_API_KEY:
        print("Groq API key loaded from Colab Secrets!")
        print(f"Key starts with: {GROQ_API_KEY[:8]}...")
    else:
        raise ValueError("Key not found in Secrets")
except Exception as e:
    # ── METHOD 2: Manual paste (fallback)
    # If Secrets is not set up, paste your key here:
    GROQ_API_KEY = "paste-your-groq-key-here"   # ← replace if not using Secrets
    if GROQ_API_KEY == "paste-your-groq-key-here":
        print("ERROR: Key not found in Secrets and not pasted manually.")
        print("")
        print("Option A: Add to Colab Secrets:")
        print("  1. Click the key icon in the left sidebar")
        print("  2. Add secret name: GROQ_API_KEY")
        print("  3. Paste your key as the value")
        print("  4. Enable Notebook access and re-run this cell")
        print("")
        print("Option B: Paste your key directly above (replace 'paste-your-groq-key-here')")
        print("Get your free key from: https://console.groq.com")
    else:
        print(f"Groq API key set manually. Starts with: {GROQ_API_KEY[:8]}...")

# Set as environment variable so LangChain reads it automatically
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Final check
if len(GROQ_API_KEY) > 20:
    print("Ready to proceed!")


## Step 2 — Install Required Packages
Run this once per session. Takes about 30 seconds.

In [ ]:
# ── INSTALL PACKAGES ──────────────────────────────────────────
# langchain-groq    → connects LangChain to Groq API
# langchain         → the main AI agent framework
# langchain-community → extra LangChain tools and utilities
# -q means quiet (hides detailed logs so output stays clean)

print("Installing packages... (takes ~30 seconds)")
!pip install -q langchain-groq langchain langchain-community
print("\nPackages installed!")
print("  langchain-groq      → Groq API connector")
print("  langchain           → AI agent framework")
print("  langchain-community → extra tools")

## Step 3 — Understand the Concept (read before running)

### What is a prompt?
A **prompt** is the instruction you give to the AI. Think of it like a question or task description.

A good prompt has three parts:
1. **Role** → tell the AI what expert to be (e.g. 'You are a Python tutor')
2. **Task** → what to do (e.g. 'Explain what IndexError means')
3. **Constraints** → any limits (e.g. 'Keep it under 100 words')

### What is temperature?
Temperature controls how creative or precise the AI is:
- `temperature=0.0` → very precise, same answer every time (good for code)
- `temperature=0.7` → balanced (good for explanations)
- `temperature=1.0` → very creative, different each time

### What is .invoke()?
`llm.invoke(prompt)` sends the prompt to Groq and waits for the response.
It returns an `AIMessage` object. Use `.content` to get the text.

In [ ]:
# ── CONCEPT DEMO: Smallest possible LLM call ──────────────────
# Before the full code, let us see the absolute minimum needed.
# Read each line carefully.

# Import the ChatGroq class from LangChain
# This class is the bridge between Python and the Groq cloud service
from langchain_groq import ChatGroq

# Create the LLM object
# - model='llama-3.1-8b-instant': use Llama 3 with 8B parameters, 8192 token context
# - temperature=0: give a precise, consistent answer
mini_llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

# Send a simple question to the AI
# .invoke() sends the text to Groq and waits for the reply
print("Sending question to Llama 3 via Groq...")
answer = mini_llm.invoke("What is 2 + 2? Reply in exactly 3 words.")

# answer is an AIMessage object
# answer.content gives us the text string
print(f"Llama 3 says: {answer.content}")
print()
print("That is it — 3 lines of code to talk to an AI!")
print("One import, one object, one call. Everything else builds on this.")

## Step 4 — The Full Stage 1 Code
Now let us use the same pattern for a real coding assistant task.
Read EVERY comment — each one explains WHY not just WHAT.

In [ ]:
# ============================================================
# CodePilot AI Studio — Stage 1: Hello Groq
# ============================================================
# Concept  : Talk to a cloud LLM using LangChain + Groq API
# Model    : Llama 3 (8B) running on Groq's fast LPU servers
# New here : ChatGroq, .invoke(), response.content
# ============================================================

# ── IMPORT ────────────────────────────────────────────────────
# ChatGroq is LangChain's connector to the Groq cloud service
# Without it we would need to write raw HTTP requests manually
from langchain_groq import ChatGroq

# ── STEP 1: Choose the model ──────────────────────────────────
# We define it as a constant so it is easy to change in one place
# llama-3.1-8b-instant = Llama 3 model, 8 billion params, 8192 token window
# Other free Groq models: 'llama-3.1-70b-versatile', 'mixtral-8x7b-32768'
MODEL_NAME = "llama-3.1-8b-instant"

# ── STEP 2: Create the LLM connection ─────────────────────────
# This prepares the connection — it does NOT send anything yet
# It automatically reads GROQ_API_KEY from the environment variable
# temperature=0.7 is a good balance for coding explanations
llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0.7    # 0=very precise, 1=very creative
)

# ── STEP 3: Write the prompt ───────────────────────────────────
# Triple quotes (""") let us write a multi-line string
# This prompt has all three parts: Role + Task + Constraints
prompt = """
You are a helpful Python coding assistant for beginner students.

Explain what a 'bug' is in software in simple, friendly terms.
Give one real example of a common Python bug with code.
Keep your answer under 150 words.
"""

# ── STEP 4: Send the prompt and get a response ────────────────
# .invoke() sends the prompt to Groq via the internet
# Groq runs Llama 3 on their special LPU servers
# This takes about 1-3 seconds (much faster than a local model!)
print("=" * 60)
print("CodePilot AI Studio — Stage 1: Hello Groq")
print("=" * 60)
print(f"Model  : {MODEL_NAME}")
print(f"Prompt : {prompt.strip()[:80]}...")
print("-" * 60)
print("Llama 3 says:")
print()

# THIS is the actual AI call — one line!
response = llm.invoke(prompt)

# response is an AIMessage object
# response.content is the text string the AI generated
print(response.content)
print()
print("=" * 60)
print("Stage 1 complete! Groq + LangChain is working.")
print()
print("WHAT JUST HAPPENED:")
print("  1. Python sent your prompt to Groq via the internet")
print("  2. Groq ran Llama 3 on their fast LPU hardware")
print("  3. Llama 3 generated a response token by token")
print("  4. LangChain wrapped it in an AIMessage object")
print("  5. We printed response.content to see the text")

## Step 5 — Verify It Worked

In [ ]:
# ── VERIFICATION ──────────────────────────────────────────────
try:
    if response and len(response.content) > 20:
        print("VERIFICATION PASSED")
        print(f"  Model used     : {MODEL_NAME}")
        print(f"  Response length: {len(response.content)} characters")
        print(f"  Preview        : {response.content[:80]}...")
        print()
        print("You are ready for Stage 2!")
    else:
        print("WARNING: Response looks empty. Run Step 4 again.")
except NameError:
    print("ERROR: Run Step 4 first before verifying.")

## Step 6 — Try It Yourself (3 Experiments)
**Change one thing at a time and observe what changes.**

In [ ]:
# ── EXPERIMENT 1: Change the prompt ───────────────────────────
# The simplest way to learn how LLMs work — change what you ask.
# Replace the text below and run this cell.

my_prompt = """
You are a Python tutor for beginners.
Write a simple Python function that checks if a number is even or odd.
Add a comment on EVERY single line explaining what it does.
"""

result = llm.invoke(my_prompt)
print("=== My Custom Prompt Result ===")
print(result.content)
print()
print("Try changing my_prompt to ask something different!")
print("Ideas: explain a loop, write a function, explain an error message")

In [ ]:
# ── EXPERIMENT 2: Compare temperature=0 vs temperature=1 ─────
# Run this cell and compare the two outputs.
# They use the SAME prompt but different temperatures.

same_prompt = "Write a Python function to reverse a string. Keep it short."

llm_precise  = ChatGroq(model=MODEL_NAME, temperature=0.0)
llm_creative = ChatGroq(model=MODEL_NAME, temperature=1.0)

r_precise  = llm_precise.invoke(same_prompt)
r_creative = llm_creative.invoke(same_prompt)

print("=== Temperature = 0.0 (precise) ===")
print(r_precise.content)
print()
print("=== Temperature = 1.0 (creative) ===")
print(r_creative.content)
print()
print("Question: What is different? Run this cell again — does temp=0 give the same answer?")

In [ ]:
# ── EXPERIMENT 3: Look inside the response object ─────────────
# response.content is just the text.
# But the response object contains much more useful information.

r = llm.invoke("Say exactly: Stage 1 experiment complete.")

print("=== Full Response Object ===")
print(f"Type                : {type(r).__name__}")
print(f"Content (text)      : {r.content}")
print(f"Model used          : {r.response_metadata.get('model', 'unknown')}")
print(f"Token usage         : {r.response_metadata.get('token_usage', {})}")
print()
print("'token_usage' tells you how many tokens were used.")
print("Groq free tier allows thousands of tokens per minute.")
print("Each student has their own limit — no sharing issues in class!")

## Summary — Stage 1 Complete!

| Concept | What it means |
|---------|---------------|
| **LLM** | Large Language Model — the AI brain |
| **Groq** | Free cloud service running LLMs super fast |
| **Llama 3** | Open-source AI model by Meta |
| **LangChain** | Python framework connecting code to AI |
| **`ChatGroq`** | LangChain class that talks to Groq |
| **Prompt** | Instruction you give the AI: Role + Task + Constraints |
| **Temperature** | Creativity: 0=precise, 1=creative |
| **`.invoke()`** | Send prompt → wait → return AIMessage |
| **`response.content`** | The text the AI generated |

---
### Problem with Stage 1
Every LLM call is completely **independent** — the AI forgets the previous message.
Ask `'fix that bug'` → AI says `'which bug?'` — it has no memory!

**Stage 2 solves this with short-term conversation memory.**

➡️ Open `stage2_memory_agent/Stage2_Memory_Agent.ipynb`